<a href="https://colab.research.google.com/github/sw030701-ai/motor-control-optimization/blob/main/experiments/02_pid_baseline_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Baseline PID Tuning Experiment
### Fixed Nominal Motor Plant → Sequential Manual PID Tuning → Baseline Record

---

### Overview

이 notebook은 `01_dc_motor_model.ipynb`에서 정한 nominal motor plant와 reference speed를 사용해 **Conventional PID Baseline**을 만든다.

```text
Fixed DC Motor Plant
      ↓
Fixed Step Reference Speed
      ↓
Kp Scan
      ↓
Ki Scan
      ↓
Small Kd Scan
      ↓
Baseline PID Gains Fix
      ↓
J_baseline 계산 및 record 저장
```

여기서는 motor parameter 선정과 open-loop validation을 반복하지 않는다. 그 내용은 `01_dc_motor_model.ipynb`가 담당한다.

In [ ]:
import os, sys, json, platform, subprocess, math
from pathlib import Path


def _in_colab():
    return "google.colab" in sys.modules


def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").exists() and (cand / "docs").exists():
            return cand
    return p


REPO_URL = "https://github.com/sw030701-ai/motor-control-optimization.git"

if _in_colab():
    root = Path("/content/motor-control-optimization")
    if not root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(root)], check=True)
    else:
        subprocess.run(["git", "pull", "--ff-only"], cwd=root, check=False)
    ROOT = root
else:
    ROOT = _find_root(Path.cwd())

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/xdgcache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
if not _in_colab():
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False


def _git(*args):
    try:
        return subprocess.check_output(["git", *args], cwd=ROOT, text=True).strip()
    except Exception:
        return None

ENV = {
    "root": str(ROOT),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "git_commit": _git("rev-parse", "HEAD"),
    "git_dirty": bool(_git("status", "--short")),
}

print(json.dumps(ENV, indent=2, ensure_ascii=False))

In [ ]:
SAVE_ARTIFACTS = True
SHOW_SCAN_TABLES = True

RESULT_TABLE_DIR = Path("results") / "tables"
RESULT_FIGURE_DIR = Path("results") / "figures"
if SAVE_ARTIFACTS:
    RESULT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
    RESULT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("SAVE_ARTIFACTS:", SAVE_ARTIFACTS)
print("SHOW_SCAN_TABLES:", SHOW_SCAN_TABLES)

## Section 1 — Load Fixed Plant and Reference

- **Plant**: `src.motor.dc_motor.nominal_dc_motor_params()`에 정의된 literature-based nominal motor.
- **Reference**: `01_dc_motor_model.ipynb`에서 정한 reachable step reference.

만약 `01` 결과 파일이 아직 없으면, 같은 공식으로 reference를 다시 계산한다.

In [ ]:
from src.motor.dc_motor import NOMINAL_MOTOR_SOURCE, nominal_dc_motor_params
from src.simulation.pid_simulation import reference_from_reachable_speed

params = nominal_dc_motor_params()
V_MAX = 12.0
REFERENCE_FILE = RESULT_TABLE_DIR / "reference_selection_record.json"

if REFERENCE_FILE.exists():
    reference_record = json.loads(REFERENCE_FILE.read_text(encoding="utf-8"))
    OMEGA_REF = float(reference_record["omega_ref_rad_s"])
else:
    reference_record = {
        "V_max": V_MAX,
        "omega_ss_max_rad_s": params.no_load_steady_state_speed(V_MAX),
        "reference_fraction": 0.50,
        "omega_ref_rad_s": round(reference_from_reachable_speed(params, V_MAX, fraction=0.50), 2),
        "note": "Recomputed because 01 reference record was not found.",
    }
    OMEGA_REF = float(reference_record["omega_ref_rad_s"])

plant_summary = pd.DataFrame([{
    "R": params.R,
    "L": params.L,
    "J_m": params.J_m,
    "b": params.b,
    "K_t": params.K_t,
    "K_e": params.K_e,
    "V_max": V_MAX,
    "omega_ref": OMEGA_REF,
}])

display(plant_summary)
print(json.dumps(reference_record, indent=2, ensure_ascii=False))

## Section 2 — Cost Function and Acceptance Criteria

Baseline tuning은 optimizer가 아니다. 여기서는 사람이 manual tuning할 때의 순서를 reproducible하게 기록한다.

Cost는 baseline을 선택한 뒤 benchmark로 계산한다.

```math
\mathcal{J}
=
0.60\mathcal{J}_{tracking}
+
0.25\mathcal{J}_{overshoot}
+
0.15\mathcal{J}_{control}
```

In [ ]:
COST_WEIGHTS = {
    "tracking": 0.60,
    "overshoot": 0.25,
    "control": 0.15,
}

ACCEPTANCE = {
    "overshoot_percent_max": 10.0,
    "steady_state_error_percent_max": 2.0,
    "persistent_saturation": "avoid",
    "sustained_oscillation": "avoid",
}

print("Cost weights:")
print(json.dumps(COST_WEIGHTS, indent=2, ensure_ascii=False))
print("\nAcceptance criteria:")
print(json.dumps(ACCEPTANCE, indent=2, ensure_ascii=False))

## Section 3 — Sequential Manual PID Tuning

Manual baseline tuning 순서는 다음과 같다.

```text
1. Ki = 0, Kd = 0으로 두고 Kp scan
2. Kp를 고정하고 Ki scan
3. 필요하면 작은 Kd scan
4. v1 acceptance criteria를 만족하면 baseline gains 고정
```

이 단계에서 여러 gain 후보를 넣고, 각 후보마다 closed-loop response를 simulation해서 표의 metric을 계산한다.

In [ ]:
from src.simulation.baseline_tuning import BaselineTuningConfig, sequential_baseline_tuning

config = BaselineTuningConfig(
    omega_ref=OMEGA_REF,
    V_max=V_MAX,
    simulation_time=10.0,
    dt=0.001,
    settling_target=2.0,
)

tuning = sequential_baseline_tuning(params, config)

kp_scan = pd.DataFrame(tuning["Kp_scan"])
ki_scan = pd.DataFrame(tuning["Ki_scan"])
kd_scan = pd.DataFrame(tuning["Kd_scan"])
selected = tuning["selected"]
final_gains = tuning["final_gains"]

columns = [
    "K_p", "K_i", "K_d", "total", "tracking", "overshoot", "overshoot_cost", "control",
    "overshoot_percent", "settling_time", "steady_state_error_percent",
    "voltage_max_abs", "saturation_percent", "omega_final",
]

print("Selected baseline gains:")
print(final_gains)

In [ ]:
if SHOW_SCAN_TABLES:
    print("Kp scan")
    display(kp_scan[columns].round(6))

**Kp 선택 기준 요약**

$K_i=0$, $K_d=0$으로 고정한 P-only scan에서 다음 조건을 처음 만족하는 값을 고른다.

```text
ω_final >= 0.60 × ω_ref
overshoot <= 5%
saturation_percent < 1%
```

In [ ]:
if SHOW_SCAN_TABLES:
    print("Ki scan")
    display(ki_scan[columns].round(6))

**Ki 선택 기준 요약**

$K_p$를 고정한 뒤 $K_i$를 증가시켜 steady-state error를 제거한다.

```text
steady-state error <= 2%
settling time <= 2.0 s
overshoot <= 10%
persistent saturation 없음
```

In [ ]:
if SHOW_SCAN_TABLES:
    print("Kd scan")
    display(kd_scan[columns].round(6))

**Kd 선택 기준 요약**

$K_d$는 꼭 큰 값을 쓸 필요가 없다. v1에서는 안정 조건을 만족하는 후보 중 settling time과 overshoot가 좋은 작은 derivative gain을 선택한다.

## Section 4 — Baseline Closed-Loop Simulation

선택된 baseline gains를 고정하고 closed-loop response를 다시 계산한다.

```text
ω_ref(t)
      ↓
e(t) = ω_ref(t) - ω(t)
      ↓
PID Controller
      ↓
V(t)
      ↓
DC Motor Plant
      ↓
ω(t)
```

In [ ]:
from src.optimization.cost_function import accepted_baseline, compute_cost
from src.simulation.pid_simulation import simulate_pid

baseline_result = simulate_pid(
    motor_params=params,
    gains=final_gains,
    omega_ref=OMEGA_REF,
    V_max=V_MAX,
    simulation_time=config.simulation_time,
    dt=config.dt,
)

baseline_cost = compute_cost(baseline_result, omega_ref=OMEGA_REF, V_max=V_MAX)

baseline_record = pd.DataFrame([{
    "K_p_baseline": final_gains.K_p,
    "K_i_baseline": final_gains.K_i,
    "K_d_baseline": final_gains.K_d,
    "reference_speed_rad_s": OMEGA_REF,
    "V_max": V_MAX,
    "J_baseline": baseline_cost["total"],
    "J_tracking": baseline_cost["tracking"],
    "J_overshoot": baseline_cost["overshoot_cost"],
    "J_control": baseline_cost["control"],
    "overshoot_percent": baseline_cost["overshoot_percent"],
    "steady_state_error_percent": baseline_cost["steady_state_error_percent"],
    "settling_time_s": baseline_cost["settling_time"],
    "max_abs_voltage": baseline_cost["voltage_max_abs"],
    "saturation_percent": baseline_cost["saturation_percent"],
    "accepted_v1": accepted_baseline(baseline_cost),
}])

display(baseline_record.T.rename(columns={0: "value"}))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

axes[0].plot(baseline_result["time"], baseline_result["omega"], label="Baseline PID speed")
axes[0].axhline(OMEGA_REF, linestyle="--", color="tab:red", label="Reference")
axes[0].set_ylabel("omega [rad/s]")
axes[0].set_title("Baseline PID Closed-Loop Response")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(baseline_result["time"], baseline_result["error"], color="tab:orange")
axes[1].axhline(0.0, linestyle="--", color="black", linewidth=1)
axes[1].set_ylabel("error [rad/s]")
axes[1].grid(True, alpha=0.3)

axes[2].plot(baseline_result["time"], baseline_result["voltage"], color="tab:green")
axes[2].axhline(V_MAX, linestyle="--", color="tab:red", linewidth=1)
axes[2].axhline(-V_MAX, linestyle="--", color="tab:red", linewidth=1)
axes[2].set_xlabel("Time [s]")
axes[2].set_ylabel("voltage [V]")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
if SAVE_ARTIFACTS:
    plt.savefig(RESULT_FIGURE_DIR / "baseline_pid_response.png", dpi=160)
plt.show()

## Section 5 — Tuning Record Export

Baseline tuning record를 `results/tables/`에 저장한다.

```text
baseline_pid_tuning_record.csv
baseline_pid_tuning_record.md
baseline_pid_tuning_record.json
baseline_kp_scan.csv
baseline_ki_scan.csv
baseline_kd_scan.csv
```

In [ ]:
record = baseline_record.iloc[0].to_dict()
record["motor_parameters"] = {
    "R": params.R,
    "L": params.L,
    "J_m": params.J_m,
    "b": params.b,
    "K_t": params.K_t,
    "K_e": params.K_e,
}
record["motor_source"] = NOMINAL_MOTOR_SOURCE
record["environment"] = ENV

if SAVE_ARTIFACTS:
    baseline_record.to_csv(RESULT_TABLE_DIR / "baseline_pid_tuning_record.csv", index=False)
    md_rows = ["| Item | Value |", "|---|---:|"]
    for key, value in baseline_record.iloc[0].items():
        if isinstance(value, float):
            text = f"{value:.10g}"
        else:
            text = str(value)
        md_rows.append(f"| `{key}` | {text} |")
    (RESULT_TABLE_DIR / "baseline_pid_tuning_record.md").write_text(
        "\n".join(md_rows) + "\n",
        encoding="utf-8",
    )
    with open(RESULT_TABLE_DIR / "baseline_pid_tuning_record.json", "w", encoding="utf-8") as f:
        json.dump(record, f, indent=2, ensure_ascii=False)
    kp_scan.to_csv(RESULT_TABLE_DIR / "baseline_kp_scan.csv", index=False)
    ki_scan.to_csv(RESULT_TABLE_DIR / "baseline_ki_scan.csv", index=False)
    kd_scan.to_csv(RESULT_TABLE_DIR / "baseline_kd_scan.csv", index=False)

print("Baseline tuning record saved." if SAVE_ARTIFACTS else "SAVE_ARTIFACTS=False, no files saved.")
print(json.dumps({
    "K_p": final_gains.K_p,
    "K_i": final_gains.K_i,
    "K_d": final_gains.K_d,
    "J_baseline": baseline_cost["total"],
    "overshoot_percent": baseline_cost["overshoot_percent"],
    "steady_state_error_percent": baseline_cost["steady_state_error_percent"],
    "settling_time_s": baseline_cost["settling_time"],
    "accepted_v1": accepted_baseline(baseline_cost),
}, indent=2, ensure_ascii=False))

## Final Summary

이번 notebook에서 baseline PID gains는 다음과 같이 고정한다.

```text
K_p_baseline = 0.80
K_i_baseline = 2.00
K_d_baseline = 0.002
```

이 값은 `Manual PID → Random Search → Bayesian Optimization → RL Controller` 비교에서 conventional PID benchmark로 사용한다.

> **주의**: 이 baseline은 intentionally poor baseline이 아니다. 안정적이고 납득 가능한 conventional PID를 기준점으로 세워야 이후 optimization 성능 비교가 왜곡되지 않는다.